# LoL 10분 승패 예측 모델 — 테스트 노트북 (JupyterHub용)

이 노트북 **하나만** 올리면 됩니다. 모델 파일이 없어도 되고, 필요한 건 인터넷과 `pandas`·`matplotlib`뿐입니다.

| 모드 | 무엇을 테스트하나 | 준비물 |
|---|---|---|
| `api` (기본) | 팀 서버에 배포된 실제 서비스 **https://p4.sumzip.com** 의 예측 API | 없음 |
| `local` | GitHub 저장소를 받아 `predict.py`·`artifacts/model.joblib` 을 직접 실행 | Python 3.11 + scikit-learn 1.9.0 (없으면 자동으로 `api` 로 되돌아감) |

두 모드는 **같은 `predict.py`** 를 쓰므로 결과가 같아야 합니다 (서빙 파리티). 마지막 셀에서 그것도 확인합니다.

입력은 13개 "블루 − 레드" 차이값이고 **양수면 블루 우세**입니다. 승리요인 해석 주의: 계수는 골드를 통제한 뒤의 값이라 `KillsDiff` 가 음수로 나와도 "킬하면 진다"가 아니라 "킬로 번 돈이 이미 골드차에 들어 있다"는 뜻입니다.


In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────────
MODE = "api"                      # "api" 또는 "local"
BASE_URL = "https://p4.sumzip.com"   # 팀 서버 (프런트 9504 가 /api 를 백엔드 9524 로 중계)
REPO_URL = "https://github.com/wpalswpa/project2608.git"
REPO_DIR = "project2608"

import json, os, subprocess, sys, urllib.request, urllib.error
import pandas as pd

FEATURES = ["FirstBlood","KillsDiff","GoldDiff","ExpDiff","WardsPlacedDiff","WardsDestroyedDiff",
            "AssistsDiff","DragonsDiff","HeraldsDiff","TowersDestroyedDiff","AvgLevelDiff",
            "TotalMinionsKilledDiff","TotalJungleMinionsKilledDiff"]

def api(path, data=None, timeout=30):
    """팀 서버 API 호출. data 가 있으면 POST."""
    req = urllib.request.Request(BASE_URL + path, data=json.dumps(data).encode() if data is not None else None,
                                 headers={"Content-Type": "application/json"}, method="POST" if data is not None else "GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}: {e.read().decode(errors='replace')[:300]}"}

def predict_api(payload):
    return api("/api/predict", payload)

_local_predict = None
def predict_local(payload):
    """저장소를 받아 predict.py 를 직접 실행. 환경이 안 맞으면 명확한 오류를 낸다."""
    global _local_predict
    if _local_predict is None:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        sys.path.insert(0, os.path.abspath(REPO_DIR))
        from predict import predict as _p          # 예측 로직의 단일 진실
        _local_predict = _p
    return _local_predict(payload)

def predict(payload):
    if MODE == "local":
        try:
            return predict_local(payload)
        except Exception as e:
            print(f"[local 실패 → api 로 전환] {type(e).__name__}: {str(e)[:200]}\n"
                  "  (모델은 scikit-learn 1.9.0 · Python 3.11 저장본입니다. 허브 커널이 다르면 local 은 안 됩니다.)")
    return predict_api(payload)

def show(result, title=None):
    """예측 결과를 사람이 읽게 출력"""
    if "error" in result:
        print("⚠", result["error"]); return
    if title: print(f"=== {title} ===")
    print(f"블루 승리 확률 {result['win_prob_blue']:.1%} → {result['pred_label']}")
    for f in result["top_factors"]:
        print(f"  {'▲' if f['contribution'] > 0 else '▼'} {f['name']:<14} = {f['value']:>8}  ({f['direction']}, 기여 {f['contribution']:+.3f})")
    for w in result.get("warnings", []):
        print("  ⚠", w)
    m = result.get("meta", {})
    print(f"  [{m.get('model')} v{m.get('version')} · {m.get('time_point_min')}분 모델 · 홀드아웃 정확도 {m.get('holdout_accuracy')}]")

print("MODE =", MODE, "| BASE_URL =", BASE_URL)


## 1. 서버 상태 — 모델이 올라가 있고 파리티가 통과됐는지

In [ ]:
h = api("/api/health")
if "error" in h:
    print("⚠ 서버에 연결할 수 없습니다:", h["error"])
else:
    print("status:", h["status"], "| domain:", h["domain"], "| 시작:", h["started_at"])
    print("model :", h["model"])
    print("parity:", h["parity"]["passed"], "(max_abs_diff =", h["parity"]["max_abs_diff"], ")")


## 2. 예시 3건 — 접전 / 블루 우세 / 레드 우세
서버가 가진 예시(`/api/examples`)를 그대로 돌립니다. 기대값: **51.2% / 94.6% / 16.8%**

In [ ]:
examples = api("/api/examples")
rows = []
for ex in examples:
    r = predict(ex["payload"])
    show(r, ex["label"]); print()
    rows.append({"예시": ex["label"], "승률": r["win_prob_blue"], "예측": r["pred_label"], "경고 수": len(r["warnings"])})
pd.DataFrame(rows)


## 3. 직접 입력해서 테스트
아래 값을 바꾸고 실행하세요. 학습 범위(대략 골드 ±5,900 · 경험치 ±4,650)를 벗어나면 예측은 나오되 **경고**가 붙습니다.

In [ ]:
my_game = dict(
    FirstBlood=1,          # 첫 킬을 블루가 했나 (1/0)
    KillsDiff=3,           # 킬 차이
    GoldDiff=1800,         # 골드 차이  ← 가장 중요
    ExpDiff=900,           # 경험치 차이 ← 2위
    WardsPlacedDiff=2, WardsDestroyedDiff=1,
    AssistsDiff=4,
    DragonsDiff=1,         # 드래곤 차이 ← 3위 (-1/0/1)
    HeraldsDiff=0, TowersDestroyedDiff=0,
    AvgLevelDiff=0.4,
    TotalMinionsKilledDiff=12, TotalJungleMinionsKilledDiff=3,
)
show(predict(my_game), "내 입력")


## 4. 여러 시나리오 한 번에 (일괄 예측)
`/api/predict/batch` 는 단건 경로를 그대로 재사용하므로 같은 입력엔 같은 답이 나옵니다.

In [ ]:
base = {f: 0 for f in FEATURES}
scenarios = {
    "완전 팽팽":            {},
    "골드만 +2000":          {"GoldDiff": 2000},
    "골드 +2000, 경험치 -1500": {"GoldDiff": 2000, "ExpDiff": -1500},
    "킬만 +5 (골드 0)":      {"KillsDiff": 5},
    "드래곤만 +1":           {"DragonsDiff": 1},
    "와드만 +50":            {"WardsPlacedDiff": 50},
    "레드 압도 (-5000)":     {"GoldDiff": -5000, "ExpDiff": -3000, "KillsDiff": -6},
}
payloads = [{**base, **v} for v in scenarios.values()]
results = api("/api/predict/batch", payloads) if MODE == "api" else [predict(p) for p in payloads]
df = pd.DataFrame([{"시나리오": k, "블루 승률": r["win_prob_blue"], "예측": r["pred_label"],
                    "1위 요인": r["top_factors"][0]["name"], "경고": len(r["warnings"])} for k, r in zip(scenarios, results)])
df


## 5. 민감도 — 골드차·경험치차만 움직이면 승률이 어떻게 변하나
로지스틱 회귀라 S자 곡선이 나와야 정상입니다. ±1,000 안이 '접전 구간'(사실상 50%)입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager

# 허브에 한글 폰트가 없으면 글자가 □ 로 깨진다 → 있으면 쓰고, 없으면 영어 라벨로
_ko = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "NanumBarunGothic"]
            if any(f == x.name for x in font_manager.fontManager.ttflist)), None)
if _ko: plt.rcParams["font.family"] = _ko
plt.rcParams["axes.unicode_minus"] = False
L = (lambda ko, en: ko if _ko else en)

def sweep(feature, values):
    payloads = [{**{f: 0 for f in FEATURES}, feature: float(v)} for v in values]
    res = api("/api/predict/batch", payloads) if MODE == "api" else [predict(p) for p in payloads]
    return [r["win_prob_blue"] for r in res]

gold = np.arange(-6000, 6001, 500); exp = np.arange(-4500, 4501, 500)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(gold, sweep("GoldDiff", gold), marker="o"); ax[0].axvspan(-1000, 1000, alpha=.15, color="orange", label=L("접전 구간", "close game"))
ax[0].set_title(L("GoldDiff → 블루 승률", "GoldDiff → blue win prob")); ax[0].set_xlabel(L("골드 차이 (블루−레드)", "gold diff (blue−red)")); ax[0].axhline(.5, ls="--", c="gray"); ax[0].legend()
ax[1].plot(exp, sweep("ExpDiff", exp), marker="o", color="green"); ax[1].set_title(L("ExpDiff → 블루 승률", "ExpDiff → blue win prob")); ax[1].set_xlabel(L("경험치 차이", "experience diff")); ax[1].axhline(.5, ls="--", c="gray")
for a in ax: a.set_ylim(0, 1); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 서버 리포트 — 성적표 · 승리요인 순위 · 언제 틀리나 · 경기 유형
숫자는 전부 서버 산출물(`reports/*.csv`)이고 노트북은 표시만 합니다.

In [ ]:
rep = api("/api/report"); mt = api("/api/match-types")
perf = rep["performance"]
flat = {}
for k, v in perf.items():
    if isinstance(v, dict):
        for k2, v2 in v.items(): flat[f"{k}.{k2}"] = v2
    else: flat[k] = v
print("■ 성적표"); display(pd.Series(flat).to_frame("값"))
print("■ 승리요인 순위 (계수 × permutation 교차)"); display(pd.DataFrame(rep["win_factors"]).head(6))
print("■ 언제 틀리나 — 골드차 구간별 정확도  (가장 약한 구간:", rep["errors"]["weakest_bin"], ")"); display(pd.DataFrame(rep["errors"]["bins"]))
print("■ 경기 유형 (k =", mt["k"], ") —", mt["note"])
display(pd.DataFrame([{"유형": t["label"], "경기수": t["count"], "비중%": t["share_pct"], "리드팀 최종승률": t["lead_team_win_rate"], **t["centroid"]} for t in mt["types"]]))


## 7. 서빙 파리티 — API 결과와 로컬 `predict.py` 결과가 같은가
`local` 환경이 되는 허브에서만 의미가 있습니다. 안 되면 건너뜁니다.

In [ ]:
try:
    diffs = []
    for ex in api("/api/examples"):
        a = predict_api(ex["payload"])["win_prob_blue"]; l = predict_local(ex["payload"])["win_prob_blue"]
        diffs.append(abs(a - l)); print(f"{ex['label']:<12} api {a:.4f}  local {l:.4f}  diff {abs(a-l):.6f}")
    print("파리티", "통과 ✅" if max(diffs) == 0 else f"⚠ 최대 차이 {max(diffs)}")
except Exception as e:
    print("local 실행 불가 →", type(e).__name__, str(e)[:160])
    print("(허브 커널이 Python 3.11 + scikit-learn 1.9.0 이어야 합니다. api 모드 결과는 위에서 이미 확인됐습니다.)")


## 8. (선택) 팀 DB 홀드아웃으로 정확도 재측정
팀 DB 비밀번호가 있을 때만. `os.environ["DB_PASSWORD"]` 를 채우고 실행하면 `v_diff13_test`(1,976판)로 정확도를 다시 잽니다.
기대값: 이 분할(`ml_split`) 기준 **약 0.71**, 문서의 0.7394 는 다른 분할(시드 42) 값 — 분할이 다르면 값이 다릅니다.


In [ ]:
import os
if not os.environ.get("DB_PASSWORD"):
    print("DB_PASSWORD 가 없어 건너뜁니다. 예:  os.environ['DB_PASSWORD'] = '...'  후 다시 실행")
else:
    import pymysql
    con = pymysql.connect(host=os.environ.get("DB_HOST", "localhost"), port=int(os.environ.get("DB_PORT", 3306)),
                          user=os.environ.get("DB_USER", "root"), password=os.environ["DB_PASSWORD"],
                          database=os.environ.get("DB_NAME", "lol_db"), connect_timeout=15)
    test = pd.read_sql("SELECT * FROM v_diff13_test ORDER BY gameId", con); con.close()
    payloads = test[FEATURES].astype(float).to_dict("records")
    res = []
    for i in range(0, len(payloads), 500):           # 500건씩 일괄
        res += api("/api/predict/batch", payloads[i:i+500]) if MODE == "api" else [predict(p) for p in payloads[i:i+500]]
    pred = pd.Series([r["pred"] for r in res]); acc = (pred.values == test["y"].values).mean()
    print(f"홀드아웃 {len(test)}판 정확도: {acc:.4f}")
    g = test["GoldDiff"].abs(); hit = (pred.values == test["y"].values)
    for lo, hi, lab in [(0,1000,"접전"),(1000,2500,"우세"),(2500,4200,"크게 우세"),(4200,1e9,"결정")]:
        m = (g >= lo) & (g < hi); print(f"  {lab:<6} {m.sum():>5}판  정확도 {hit[m].mean():.3f}")
